In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import os
os.chdir(r'G:\My Drive\Study\Time Series Analysis Udemy\Data\time series analysis forecasting')
os.getcwd()


'G:\\My Drive\\Study\\Time Series Analysis Udemy\\Data\\time series analysis forecasting'

In [4]:
import itertools # looping through all possible combinations of the options we want to try

In [14]:
from sklearn.metrics import mean_squared_error

In [5]:
df = pd.read_csv(r'airline_passengers.csv', index_col = 'Month', parse_dates = True)

In [6]:
df.index

DatetimeIndex(['1949-01-01', '1949-02-01', '1949-03-01', '1949-04-01',
               '1949-05-01', '1949-06-01', '1949-07-01', '1949-08-01',
               '1949-09-01', '1949-10-01',
               ...
               '1960-03-01', '1960-04-01', '1960-05-01', '1960-06-01',
               '1960-07-01', '1960-08-01', '1960-09-01', '1960-10-01',
               '1960-11-01', '1960-12-01'],
              dtype='datetime64[ns]', name='Month', length=144, freq=None)

In [7]:
df.index.freq = 'MS'

In [8]:
df.index

DatetimeIndex(['1949-01-01', '1949-02-01', '1949-03-01', '1949-04-01',
               '1949-05-01', '1949-06-01', '1949-07-01', '1949-08-01',
               '1949-09-01', '1949-10-01',
               ...
               '1960-03-01', '1960-04-01', '1960-05-01', '1960-06-01',
               '1960-07-01', '1960-08-01', '1960-09-01', '1960-10-01',
               '1960-11-01', '1960-12-01'],
              dtype='datetime64[ns]', name='Month', length=144, freq='MS')

In [9]:
df.shape

(144, 1)

In [10]:
h = 12 # forecast horizon
steps = 10 #validating over 10 steps using walk forward validation
Ntest = len(df) - h - steps + 1

In [12]:
#hyperparameters to try
trend_type_list = ['add','mul']
seasonal_type_list = ['add','mul']
damped_trend_list = [True, False] # enables dampening of trend component
init_method_list = ['estimated','heuristic','legacy-heuristic']
use_boxcox_list = [True, False, 0] 
#passing 0 as float tells the value of lambda, since lambda value we are trying is 0, it will be log transformation
# true means auto lambda optimization

## implementing walkforward function

In [18]:
def walkforward(trend_type, seasonal_type, damped_trend, init_method, use_boxcox, debug = False):
    #store errors
    errors = []
    seen_last = False
    steps_completed = 0

    for end_of_train in range(Ntest, len(df) - h + 1):
        train = df.iloc[:end_of_train]
        test = df.iloc[end_of_train : end_of_train + h]

        if test.index[-1] == df.index[-1]:
            seen_last = True

        steps_completed += 1

        hw = ExponentialSmoothing(
            train['Passengers'],
            initialization_method = init_method,
            trend = trend_type,
            damped_trend = damped_trend,
            seasonal = seasonal_type,
            seasonal_periods = 12,
            use_boxcox =  use_boxcox)
        
        res_hw = hw.fit()

        fcast = res_hw.forecast(h)
        error = mean_squared_error(test['Passengers'],fcast)
        errors.append(error)

    if debug:
        print("seen_last:", seen_last)
        print("steps completed:", steps_completed)

    return np.mean(errors)
    

In [19]:
# testing
walkforward('add','add',False,'legacy-heuristic', 0, debug = True )

seen_last: True
steps completed: 10


np.float64(1448.5336379633752)

In [21]:
# this is using itertools, else we had to use multiple nested for loops

# iterate through all possible options (grid search)
tuple_of_option_lists = (
    trend_type_list,
    seasonal_type_list,
    damped_trend_list,
    init_method_list,
    use_boxcox_list,
)

i = 1
for x in itertools.product(*tuple_of_option_lists):
    print(x, i)
    i += 1
    

('add', 'add', True, 'estimated', True) 1
('add', 'add', True, 'estimated', False) 2
('add', 'add', True, 'estimated', 0) 3
('add', 'add', True, 'heuristic', True) 4
('add', 'add', True, 'heuristic', False) 5
('add', 'add', True, 'heuristic', 0) 6
('add', 'add', True, 'legacy-heuristic', True) 7
('add', 'add', True, 'legacy-heuristic', False) 8
('add', 'add', True, 'legacy-heuristic', 0) 9
('add', 'add', False, 'estimated', True) 10
('add', 'add', False, 'estimated', False) 11
('add', 'add', False, 'estimated', 0) 12
('add', 'add', False, 'heuristic', True) 13
('add', 'add', False, 'heuristic', False) 14
('add', 'add', False, 'heuristic', 0) 15
('add', 'add', False, 'legacy-heuristic', True) 16
('add', 'add', False, 'legacy-heuristic', False) 17
('add', 'add', False, 'legacy-heuristic', 0) 18
('add', 'mul', True, 'estimated', True) 19
('add', 'mul', True, 'estimated', False) 20
('add', 'mul', True, 'estimated', 0) 21
('add', 'mul', True, 'heuristic', True) 22
('add', 'mul', True, 'heur

In [23]:
best_score = float('inf')
best_options = None
for x in itertools.product(*tuple_of_option_lists):
    score = walkforward(*x)

    if score < best_score:
        print("Best Score so far:", score)
        best_score = score
        best_options = x

Best Score so far: 412.817263262104
Best Score so far: 397.58728374682494
Best Score so far: 368.78738047449446
Best Score so far: 320.6640602957365
Best Score so far: 308.1357554049132


C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: Runt

Best Score so far: 303.71164998411547
Best Score so far: 299.8207455862339
Best Score so far: 261.8820534050848
Best Score so far: 249.58121676520523


C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
C:\Users\VARUN\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:84: Runt

In [24]:
print("best score:", best_score)
trend_type, seasonal_type, damped_trend, init_method, use_box_cox = best_options
print("trend_type:",trend_type)
print("seasonal_type:",seasonal_type)
print("damped_trend:",damped_trend)
print("init_method:",init_method)
print("use_box_cox:",use_box_cox)

best score: 249.58121676520523
trend_type: mul
seasonal_type: add
damped_trend: False
init_method: legacy-heuristic
use_box_cox: False
